In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
train = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/train.csv')
test = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/test.csv')

In [ ]:
Id = test['Id']

In [ ]:
# let's just use some variables
cols = ['LotFrontage', 'LotArea', 'YrSold']

In [ ]:
train = train[cols + ['SalePrice']]
test = test[cols]

In [ ]:
train.isnull().sum()

In [ ]:
test.isnull().sum()

In [ ]:
train

In [ ]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import GridSearchCV

# Prepare X and y
X = train.drop('SalePrice', axis=1)
y = train['SalePrice']

# Create pipeline
pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler()),
    ('knn', KNeighborsRegressor())
])

# Define parameter grid
param_grid = {
    'knn__n_neighbors': [3, 5, 7, 9, 11, 13, 15, 17, 19, 21, 23, 25, 27, 29, 31, 33, 35, 37],
    'knn__metric': ['euclidean', 'manhattan']
}

# Create GridSearchCV object
grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring='neg_mean_squared_log_error',
    n_jobs=-1  # Use all available cores
)

# Fit the grid search
grid_search.fit(X, y)

# Print results
print("\nGrid search results:")
print(f"Best parameters: {grid_search.best_params_}")
print(f"Best MSLE: {-grid_search.best_score_:,.2f}")

# Print detailed results for each combination
print("\nDetailed scores for each combination:")
for mean_score, params in zip(grid_search.cv_results_['mean_test_score'], 
                            grid_search.cv_results_['params']):
    k = params['knn__n_neighbors']
    metric = params['knn__metric']
    msle = -mean_score
    print(f"k={k}, metric={metric}: MSLE={msle:,.2f}")

# Use best model to make predictions on test set
predictions = grid_search.predict(test)

# Create submission DataFrame
submission = pd.DataFrame({
    'Id': Id,
    'SalePrice': predictions
})

# Save submission
submission.to_csv('submission.csv', index=False)

In [ ]:
# Visualize the GridSearchCV results
import matplotlib.pyplot as plt
import seaborn as sns

# Extract the results
results = pd.DataFrame(grid_search.cv_results_)

# Convert scores from negative to positive MSLE
results['positive_msle'] = -results['mean_test_score']

# Create a pivot table for the heatmap
pivot_df = pd.pivot_table(
    data=results,
    values='positive_msle',
    index='param_knn__n_neighbors',  # k values on y-axis
    columns='param_knn__metric',     # distance metric on x-axis
)

# Create the plot
plt.figure(figsize=(12, 8))
ax = sns.heatmap(
    pivot_df,
    annot=True,
    fmt=',.3f',
    cmap='Blues_r',      # Reversed Blues colormap (darker = lower performance/higher MSLE)
    linewidths=0.5,
    cbar_kws={'label': 'MSLE ($)'}
)

# Set title and labels
plt.title('KNN Regression: MSLE by Distance Metric and Number of Neighbors', fontsize=14)
plt.xlabel('Distance Metric', fontsize=12)
plt.ylabel('Number of Neighbors (k)', fontsize=12)

# Add a note about the best parameters
best_params = grid_search.best_params_
best_msle = -grid_search.best_score_
plt.figtext(
    0.5, 0.01,
    f"Best parameters: k={best_params['knn__n_neighbors']}, metric={best_params['knn__metric']} with MSLE: ${best_msle:,.2f}",
    ha='center', fontsize=12
)

plt.tight_layout(rect=[0, 0.05, 1, 0.95])  # Adjust layout to make room for the text at the bottom
plt.show()

# Also create a line plot to see the trend as k increases
plt.figure(figsize=(12, 6))

# Create a DataFrame suitable for line plotting
line_df = results.copy()
line_df['k'] = line_df['param_knn__n_neighbors']
line_df['metric'] = line_df['param_knn__metric']

# Plot the lines
sns.lineplot(
    data=line_df, 
    x='k', 
    y='positive_msle', 
    hue='metric', 
    marker='o',
    linewidth=2
)

plt.title('MSLE vs Number of Neighbors by Distance Metric', fontsize=14)
plt.xlabel('Number of Neighbors (k)', fontsize=12)
plt.ylabel('MSLE ($)', fontsize=12)
plt.grid(True, alpha=0.3)
plt.legend(title='Distance Metric')

# Format the y-axis with comma separator for thousands
from matplotlib.ticker import FuncFormatter
plt.gca().yaxis.set_major_formatter(FuncFormatter(lambda x, _: f'{x:,.2f}'))

plt.tight_layout()
plt.show()